# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      [WARNING] Unable to import torch, pre-compiling ops will be disabled. Please visit https://pytorch.org/ to see how to properly install torch on your system.
       [WARNING]  unable to import torch, please install it if you want to pre-compile any deepspeed ops.
      DS_BUILD_OPS=1
      Traceback (most recent call last):
        File "d:\education maga 2025 autumn\nlp\nlp_course\week02_lm\venv_py\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
        File "d:\education maga 2025 autumn\nlp\nlp_course\week02_lm\venv_py\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "d:\

  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached deepspeed-0.18.2.tar.gz (1.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


### Data Preparation

In [10]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating test split: 100%|██████████| 390965/390965 [00:00<00:00, 3741195.99 examples/s]




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [12]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [13]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [14]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map: 100%|██████████| 390965/390965 [00:16<00:00, 23767.53 examples/s]


In [15]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [16]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [17]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [21]:
#<A whole lot of YOUR CODE HERE>
from torch.utils.data import DataLoader
from transformers import default_data_collator
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


val_set = qqp_preprocessed["validation"]
val_loader = DataLoader(
    val_set, 
    batch_size=32, 
    shuffle=False, 
    collate_fn=default_data_collator,
    num_workers=4  
)


scaler = torch.cuda.amp.GradScaler(enabled=True) if torch.cuda.is_available() else None

total_correct = 0
total_samples = 0


with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):

        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                for k, v in batch.items()}
        

        with torch.cuda.amp.autocast(enabled=True if scaler else False):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                token_type_ids=batch["token_type_ids"],
            )
        

        predictions = torch.argmax(outputs.logits, dim=1)
        labels = batch["labels"]
        

        total_correct += (predictions == labels).sum().item()
        total_samples += labels.size(0)


accuracy = total_correct / total_samples
print(f"Validation Accuracy: {accuracy:.4f}")
accuracy = accuracy#<Validation accuracy, between 0 and 1>

C:\temp\ipykernel_18724\1110583726.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True) if torch.cuda.is_available() else None
Evaluating:   0%|          | 0/1264 [00:00<?, ?it/s]C:\temp\ipykernel_18724\1110583726.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True if scaler else False):
Evaluating: 100%|██████████| 1264/1264 [00:22<00:00, 56.89it/s] 

Validation Accuracy: 0.9084


In [22]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, default_data_collator
import time
import pandas as pd
from tqdm import tqdm
import os


MODELS_TO_COMPARE = [
    
    "gchhablani/bert-base-cased-finetuned-qqp",
    
    
    "textattack/bert-base-uncased-QQP",
    "cross-encoder/quora-roberta-base",
    "microsoft/deberta-v3-base",
]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def get_model_size(model_path):
    try:
        
        cache_dir = transformers.utils.hub.get_cache_dir()
        model_files = []
        
        
        for root, dirs, files in os.walk(cache_dir):
            if model_path.replace("/", "--") in root:
                for file in files:
                    if file.endswith(('.bin', '.safetensors', '.json', '.txt')):
                        model_files.append(os.path.join(root, file))
        
       
        total_size = sum(os.path.getsize(f) for f in model_files) / (1024 * 1024)
        return total_size
    except:
        
        if "bert-base" in model_path:
            return 440  
        elif "roberta-base" in model_path:
            return 500  
        elif "deberta-v3-base" in model_path:
            return 550  
        else:
            return 500  


def evaluate_model(model_name, tokenizer=None, model=None):
    print(f"\n{'='*60}")
    print(f"Evaluating model: {model_name}")
    print(f"{'='*60}")
    
    
    if tokenizer is None or model is None:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
    
    model.to(device)
    model.eval()
    
   
    def preprocess_function(examples):
        result = tokenizer(
            examples["text1"],
            examples["text2"],
            padding="max_length",
            max_length=128,
            truncation=True,
        )
        result["labels"] = examples["label"]
        return result
    
    qqp_preprocessed = qqp.map(preprocess_function, batched=True)
    val_set = qqp_preprocessed["validation"]
    
    
    val_loader = DataLoader(
        val_set, 
        batch_size=32,
        shuffle=False,
        collate_fn=default_data_collator,
        num_workers=2
    )
    
    
    total_correct = 0
    total_samples = 0
    total_time = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
      
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                    for k, v in batch.items()}
            
            
            start_time = time.time()
            
            
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                token_type_ids=batch.get("token_type_ids", None), 
            )
            
            end_time = time.time()
            total_time += (end_time - start_time)
            
            
            predictions = torch.argmax(outputs.logits, dim=1)
            labels = batch["labels"]
            
            
            total_correct += (predictions == labels).sum().item()
            total_samples += labels.size(0)
    
  
    accuracy = total_correct / total_samples
    samples_per_second = total_samples / total_time
    
    return {
        "model_name": model_name,
        "accuracy": accuracy,
        "samples_per_second": samples_per_second,
        "evaluation_time": total_time
    }


results = []


for model_name in MODELS_TO_COMPARE:
    try:
        print(f"\nLoading {model_name}...")
        
        
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
        
        
        model_size_mb = get_model_size(model_name)
        
     
        metrics = evaluate_model(model_name, tokenizer, model)
        metrics["model_size_mb"] = model_size_mb
        
        results.append(metrics)
        

        del model, tokenizer
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")
        continue


df_results = pd.DataFrame(results)


df_results = df_results.sort_values("accuracy", ascending=False)


print("\n" + "="*80)
print("COMPARISON RESULTS")
print("="*80)
print(df_results.to_string(index=False))


df_results.to_csv("model_comparison_results.csv", index=False)
print("\nResults saved to 'model_comparison_results.csv'")


print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
summary_df = df_results[["model_name", "accuracy", "samples_per_second", "model_size_mb"]].copy()
summary_df.columns = ["Model", "Accuracy", "Samples/sec", "Size (MB)"]
summary_df["Accuracy"] = summary_df["Accuracy"].apply(lambda x: f"{x:.4f}")
summary_df["Samples/sec"] = summary_df["Samples/sec"].apply(lambda x: f"{x:.1f}")
summary_df["Size (MB)"] = summary_df["Size (MB)"].apply(lambda x: f"{x:.0f}")
print(summary_df.to_string(index=False))

Using device: cuda

Loading gchhablani/bert-base-cased-finetuned-qqp...

Evaluating model: gchhablani/bert-base-cased-finetuned-qqp


Map:  77%|███████▋  | 281000/363846 [00:11<00:03, 25552.51 examples/s]Error while downloading from https://huggingface.co/gchhablani/bert-base-cased-finetuned-qqp/resolve/refs%2Fpr%2F2/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...
Map:  40%|███▉      | 155000/390965 [00:06<00:11, 20922.09 examples/s]Error while downloading from https://huggingface.co/gchhablani/bert-base-cased-finetuned-qqp/resolve/refs%2Fpr%2F2/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...
Evaluating:   0%|          | 0/1264 [00:00<?, ?it/s]Error while downloading from https://huggingface.co/gchhablani/bert-base-cased-finetuned-qqp/resolve/refs%2Fpr%2F2/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...
Evaluating:  34%|███▍      | 436/1264 [00:13<00:19, 42.34it/s]Error while download


Loading textattack/bert-base-uncased-QQP...


d:\education maga 2025 autumn\nlp\nlp_course\week02_lm\venv_py\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Алексей\.cache\huggingface\hub\models--textattack--bert-base-uncased-QQP. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Error while downloading from https://huggingface.co/gchhablani/be


Evaluating model: textattack/bert-base-uncased-QQP


...


Evaluating model: cross-encoder/quora-roberta-base


Evaluating: 100%|██████████| 1264/1264 [00:34<00:00, 36.52it/s]



Loading microsoft/deberta-v3-base...


d:\education maga 2025 autumn\nlp\nlp_course\week02_lm\venv_py\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Алексей\.cache\huggingface\hub\models--microsoft--deberta-v3-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not in

Error evaluating microsoft/deberta-v3-base: 
 requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


COMPARISON RESULTS
                              model_name  accuracy  samples_per_second  evaluation_time  model_size_mb
        textattack/bert-base-uncased-QQP  0.909077         6766.321569         5.975182            440
gchhablani/bert-base-cased-finetuned-qqp  0.908385         6641.473913         6.087504            440
        cross-encoder/quora-roberta-base  0.631833         6349.199559         6.367732            500

Results saved to 'model_comparison_results.csv'

SUMMARY TABLE
                                   Model Accuracy Samples/sec Size (MB)
        textattack/bert-base-uncased-QQP   0.9091

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [27]:

import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics.pairwise import cosine_similarity


BEST_MODEL = "textattack/bert-base-uncased-QQP"
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


def get_correct_embeddings(texts):
    embeddings = []
    
    for i in tqdm(range(0, len(texts), 32), desc="Получение эмбеддингов"):
        batch_texts = texts[i:i+32]
        
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        encoded = {k: v.to(device) for k, v in encoded.items()}
        
        with torch.no_grad():
 
            outputs = model.bert(**encoded)
            
            attention_mask = encoded["attention_mask"]
            token_embeddings = outputs.last_hidden_state
            
            
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            mean_embeddings = sum_embeddings / sum_mask
            
            
            mean_embeddings = torch.nn.functional.normalize(mean_embeddings, p=2, dim=1)
            embeddings.append(mean_embeddings.cpu().numpy())
    
    return np.vstack(embeddings)


def find_similar_questions(query_question, top_k=5, similarity_threshold=0.8):
    
   
    query_embedding = get_correct_embeddings([query_question])[0]
    
    
    similarities = cosine_similarity([query_embedding], train_embeddings)[0]
    
    
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        similarity = similarities[idx]
        if similarity >= similarity_threshold:
            results.append((unique_questions[idx], similarity))
    
    return results


def check_if_duplicate(question1, question2):
    
    encoded = tokenizer(
        question1,
        question2,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    
    encoded = {k: v.to(device) for k, v in encoded.items()}
    
    with torch.no_grad():
        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1)
        duplicate_prob = probs[0, 1].item()  
    
    return duplicate_prob


def smart_find_duplicates(query_question, top_k=5, fast_search_threshold=0.7):
   
    print(f"\nПоиск дубликатов для: '{query_question}'")
    print("-" * 60)
    
   
    print("Шаг 1: Быстрый поиск кандидатов...")
    candidates = find_similar_questions(query_question, top_k=20, similarity_threshold=0.5)
    
    if not candidates:
        print("Не найдено кандидатов для проверки")
        return []
    
   
    print("Шаг 2: Точная проверка кандидатов...")
    scored_candidates = []
    
    for candidate, similarity in tqdm(candidates, desc="Проверка кандидатов"):
        duplicate_score = check_if_duplicate(query_question, candidate)
        scored_candidates.append((candidate, duplicate_score, similarity))
    
   
    scored_candidates.sort(key=lambda x: x[1], reverse=True)
    
   
    results = []
    print("\nРезультаты:")
    print("-" * 60)
    
    for i, (candidate, duplicate_score, similarity) in enumerate(scored_candidates[:top_k]):
        status = "✓ ВЕРОЯТНЫЙ ДУБЛИКАТ" if duplicate_score > 0.5 else "✗ НЕ ДУБЛИКАТ"
        print(f"{i+1}. [{duplicate_score:.3f}] {status}")
        print(f"   Схожесть эмбеддингов: {similarity:.3f}")
        print(f"   Вопрос: '{candidate[:80]}...'")
        print()
        
        results.append((candidate, duplicate_score))
    
    return results


print("\nПересчитываю эмбеддинги правильным способом...")
train_embeddings = get_correct_embeddings(unique_questions)


test_cases = [
    "How do I control my horny emotions?",
    "What can one do after MBBS?",
    "What is the best way to learn programming?",
    "How to make money online?",
    "What is artificial intelligence?"
]



for i, question in enumerate(test_cases):
    print(f"\nТест {i+1}:")
    smart_find_duplicates(question, top_k=3)


print("\n" + "="*80)
print("ПРОВЕРКА НА РЕАЛЬНЫХ ДУБЛИКАТАХ")
print("="*80)


duplicate_pairs = []
for item in qqp["train"]:
    if item["label"] == 1 and len(item["text1"]) > 20 and len(item["text2"]) > 20:
        duplicate_pairs.append(item)
    if len(duplicate_pairs) >= 5:
        break

print(f"\nПротестировано {len(duplicate_pairs)} пар дубликатов:")
correct = 0
total = 0

for i, pair in enumerate(duplicate_pairs):
    print(f"\nПара {i+1}:")
    print(f"В1: {pair['text1'][:60]}...")
    print(f"В2: {pair['text2'][:60]}...")
    

    duplicates = smart_find_duplicates(pair["text1"], top_k=1)
    
    found = False
    if duplicates:
        top_candidate, score = duplicates[0]

        if top_candidate == pair["text2"] or score > 0.5:
            found = True
            print(f"  ✓ Модель правильно определила как дубликат (оценка: {score:.3f})")
    
    if found:
        correct += 1
    else:
        print(f"  ✗ Модель не определила как дубликат")
    
    total += 1

print(f"\nИтог: {correct} из {total} дубликатов определены правильно ({correct/total*100:.1f}%)")


def demonstrate_duplicate_search():
    
    print("\n" + "="*80)
    print("ДЕМОНСТРАЦИЯ ПОИСКА ДУБЛИКАТОВ")
    print("="*80)
    
    demo_questions = [
        ("How to learn programming fast?", "programming"),
        ("What are good ways to lose weight?", "health"),
        ("Best books for beginners in finance", "books"),
        ("How to prepare for job interview?", "career"),
        ("Tips for learning Spanish quickly", "language")
    ]
    
    for question, category in demo_questions:
        print(f"\nКатегория: {category}")
        print(f"Вопрос: '{question}'")
        
        results = smart_find_duplicates(question, top_k=2)
        
        if results:
            print(f"Найдено {len(results)} потенциальных дубликатов")
        else:
            print("Дубликаты не найдены")
        
        print("-" * 60)


demonstrate_duplicate_search()





Пересчитываю эмбеддинги правильным способом...


Получение эмбеддингов: 100%|██████████| 604/604 [00:06<00:00, 95.31it/s] 



Тест 1:

Поиск дубликатов для: 'How do I control my horny emotions?'
------------------------------------------------------------
Шаг 1: Быстрый поиск кандидатов...


Получение эмбеддингов: 100%|██████████| 1/1 [00:00<00:00, 166.67it/s]


Шаг 2: Точная проверка кандидатов...


...


Результаты:
------------------------------------------------------------
1. [0.986] ✓ ВЕРОЯТНЫЙ ДУБЛИКАТ
   Схожесть эмбеддингов: 0.937
   Вопрос: 'How do I prepare for interviews?...'

2. [0.979] ✓ ВЕРОЯТНЫЙ ДУБЛИКАТ
   Схожесть эмбеддингов: 0.927
   Вопрос: 'How do I prepare for my hr interview?...'

Найдено 2 потенциальных дубликатов
------------------------------------------------------------

Категория: language
Вопрос: 'Tips for learning Spanish quickly'

Поиск дубликатов для: 'Tips for learning Spanish quickly'
------------------------------------------------------------
Шаг 1: Быстрый поиск кандидатов...


Получение эмбеддингов: 100%|██████████| 1/1 [00:00<00:00, 134.51it/s]


Шаг 2: Точная проверка кандидатов...


Проверка кандидатов: 100%|██████████| 20/20 [00:00<00:00, 201.13it/s]


Результаты:
------------------------------------------------------------
1. [0.001] ✗ НЕ ДУБЛИКАТ
   Схожесть эмбеддингов: 0.827
   Вопрос: 'How can I improve my communication skills in english?...'

2. [0.001] ✗ НЕ ДУБЛИКАТ
   Схожесть эмбеддингов: 0.827
   Вопрос: 'How can I improve my communication skills in English?...'

Найдено 2 потенциальных дубликатов
------------------------------------------------------------


### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>